# 00 · First look at every file (EDA)

This notebook looks at every Weavve file before anything is cleaned. Nothing here changes the data.

For each file it answers three questions, following the *Data Understanding* step of CRISP-DM:

| Step | Question | What you will see |
|---|---|---|
| **Describe** | What is in the file? | Size, what each row shows, the key columns, the column table |
| **Explore** | What does the data look like? | Coverage over time, distributions, summary statistics, charts |
| **Check** | Can we trust it? | Missing values, duplicates, odd values, outliers, and whether files join up |

The checks follow common data quality dimensions:

| Dimension | Plain meaning |
|---|---|
| Completeness | Are values missing? |
| Uniqueness | Are there duplicates where there should be none? |
| Validity | Do values make sense, like no negative quantities? |
| Consistency | Do files and platforms agree with each other? |
| Coverage | Are there gaps in the dates? |

Each section ends with **What this tells us**. Section I runs the same checks on every column of every tab, so nothing is missed.
The last section lists every issue found, how serious it is, and how the team plans to handle it.
The detailed measurement of each problem is in notebook 02.

## Setup

On the team page the data is already loaded. On a laptop, put the Weavve files in `data/` first (see `data/README.md`).

In [ ]:
import sys
sys.path.insert(0, "../src")          # our own helper code lives in the src folder

import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from weavve import load, paths
from weavve.load import MARKETPLACE, WEBSITE

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (8, 3.2)
print("Data folder:", paths.DATA)

### Helper tools

Five small tools used in every section. You only need to know what they show.

| Tool | What it shows |
|---|---|
| `profile(table)` | One row per column, with its type, how many cells are missing, how many different values, and one example |
| `numbers(values)` | Count, smallest, median, average, largest, and how many values are unusually high |
| `bar_chart(counts, title)` | A bar chart of counts, with the number on each bar |
| `line_chart(table, title)` | A line per column, over time |
| `histogram(values, title)` | How values are spread out. `log=True` spreads out very uneven numbers such as money |

**Unusually high (outliers).** Take the middle half of the values. Anything more than 1.5 times that width above it is
counted as unusually high. This is the standard *1.5 × IQR* rule.

In [ ]:
GREEN, ORANGE, GREY = "#2d6a5a", "#c8702a", "#9aa5a0"


def profile(table):
    """One row per column: its type, missing cells, different values and one example."""
    result = pd.DataFrame(index=table.columns)
    result["type"] = table.dtypes.astype(str)
    result["missing"] = table.isna().sum()
    result["missing %"] = (result["missing"] / len(table) * 100).round(1)
    result["different values"] = table.nunique()
    examples = []
    for column in table.columns:
        filled = table[column].dropna()
        if len(filled) > 0:
            examples.append(str(filled.iloc[0])[:40])
        else:
            examples.append("(all missing)")
    result["example"] = examples
    return result


def numbers(values):
    """Summary statistics, plus outliers by the 1.5 x IQR rule."""
    values = pd.Series(values).dropna()
    q1, q3 = values.quantile(0.25), values.quantile(0.75)
    high_fence = q3 + 1.5 * (q3 - q1)
    return pd.Series({
        "count": format(len(values), ","),
        "smallest": format(values.min(), ",.2f"),
        "25% are below": format(q1, ",.2f"),
        "median (middle)": format(values.median(), ",.2f"),
        "average": format(values.mean(), ",.2f"),
        "75% are below": format(q3, ",.2f"),
        "largest": format(values.max(), ",.2f"),
        "unusually high (above " + format(high_fence, ",.0f") + ")": format(int((values > high_fence).sum()), ","),
    })


def tidy(ax, title, xlabel="", ylabel=""):
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.spines[["top", "right"]].set_visible(False)


def bar_chart(counts, title, xlabel="", ylabel="", sideways=False):
    """Bar chart with the count on each bar. sideways=True suits long labels."""
    if sideways:
        counts = counts.sort_values()
        ax = counts.plot(kind="barh", color=GREEN, width=0.75, figsize=(8, 0.35 * len(counts) + 1))
        for i, v in enumerate(counts.values):
            ax.annotate(" " + format(v, ","), (v, i), va="center", fontsize=8)
        tidy(ax, title, ylabel, "")
    else:
        ax = counts.plot(kind="bar", color=GREEN, width=0.75)
        ax.tick_params(axis="x", rotation=0)
        for i, v in enumerate(counts.values):
            ax.annotate(format(v, ","), (i, v), ha="center", va="bottom", fontsize=8)
        tidy(ax, title, xlabel, ylabel)
    plt.tight_layout()
    plt.show()


def line_chart(table, title, ylabel="", colors=None):
    ax = table.plot(color=colors)
    tidy(ax, title, "", ylabel)
    plt.tight_layout()
    plt.show()


def histogram(values, title, xlabel="", ylabel="count", log=False, bins=40):
    values = pd.Series(values).dropna()
    if log:
        values = values[values >= 1]                    # a log scale cannot show 0, so start at 1
        bins = np.logspace(np.log10(values.min()), np.log10(values.max()), bins)
        plt.xscale("log")
    plt.hist(values, bins=bins, color=GREEN, edgecolor="white")
    tidy(plt.gca(), title, xlabel, ylabel)
    plt.tight_layout()
    plt.show()

## A. The files at a glance

The project instructions list 11 files. This notebook covers the 7 in `data/`, which are the ones the team has.
Files **6** (LTV & RFM by cohort), **7** (customer purchase history), **10** (new vs returning customers) and
**11** (orders and tiers) are listed on eLearn but are not in `data/` yet.

For files with several tabs, rows and columns are for the main tab. The other tabs are covered in each file's section.

In [ ]:
orders = load.orders()          # file 3
products = load.skus()          # file 4
discounts = load.discounts()    # file 8, main tab
loyalty = load.loyalty()        # file 9.1
ads = load.ads()                # file 12, main tab
semrush = load.semrush()        # file 2, all months stacked
books, lockfiles = load.gsc_workbooks()   # file 1, list of workbooks


def count_tabs(key):
    """How many tabs an Excel file has, as text like '5 tabs'."""
    n = len(pd.ExcelFile(paths.path(key)).sheet_names)
    return f"{n} tab" if n == 1 else f"{n} tabs"


import openpyxl
tabs_per_book = [len(openpyxl.load_workbook(book, read_only=True).sheetnames) for book in books]
tabs_low, tabs_high = min(tabs_per_book), max(tabs_per_book)

glance = pd.DataFrame([
    # file,             format,                  tabs,                                rows,            columns
    ("1 Search Console", "folder of Excel files", f"{len(books)} files, {tabs_low} to {tabs_high} tabs each", "–", "–"),
    ("2 Semrush",        "folder of CSV files",   f"{semrush['month'].nunique()} files", len(semrush),  semrush.shape[1] - 2),
    ("3 Orders",         "Excel file",            count_tabs("orders"),                len(orders),     orders.shape[1]),
    ("4 Products",       "Excel file",            count_tabs("skus"),                  len(products),   products.shape[1]),
    ("8 Discounts",      "Excel file",            count_tabs("discounts"),             len(discounts),  discounts.shape[1]),
    ("9.1 Loyalty",      "Excel file",            count_tabs("loyalty"),               len(loyalty),    loyalty.shape[1]),
    ("12 Google Ads",    "Excel file",            count_tabs("ads"),                   len(ads),        ads.shape[1] - 1),
], columns=["file", "format", "tabs", "rows", "columns"])
glance

**How to read it.** One line per file. Search Console has no single row or column count, because its tabs differ.
Semrush shows the 17 columns in each file, not counting the `month` and `source_file` columns the notebook adds when it
stacks the files. Google Ads shows 24, not counting `kw`, a cleaned copy of each keyword that the notebook adds
(lower case, without quote marks or brackets).

## B. File 3, Orders

**Each row shows** one item inside one order. Sales, refunds and cancellations all sit in the same table.
This is the most important file for the customer audit.

Two words used below:
- An **undo row** is a row where `Order or sales reversal` is `reversal`. It cancels or refunds an earlier sale row, with negative money.
- A **cancelled order** is one where `Is canceled order` is True. Both its sale rows and its undo rows stay in the file.

### B1. Describe, size and keys

In [ ]:
SALE = orders["Order or sales reversal"] == "order"      # True on normal sale rows, False on undo rows
CANCELLED = orders["Is canceled order"]

channel_names = {WEBSITE: "Website", MARKETPLACE: "Marketplace"}
orders["channel"] = orders["Order sales channel"].map(channel_names).fillna("Other")
sales = orders[SALE & ~CANCELLED]                        # sale rows of orders that were not cancelled

lines_per_order = orders.groupby("Order ID").size()
pd.Series({
    "rows (order lines)": format(len(orders), ","),
    "different orders (Order ID)": format(orders["Order ID"].nunique(), ","),
    "different customer IDs": format(orders["Customer ID"].nunique(), ","),
    "average rows per order": round(lines_per_order.mean(), 1),
    "most rows in one order": lines_per_order.max(),
    "first day": orders["Day"].min().date(),
    "last day": orders["Day"].max().date(),
}).to_frame("value")

**How to read it.** The file has far more rows than orders, because one order with 3 items takes 3 rows.
Count orders with `Order ID`, never by counting rows.

### B1b. Check, is each order line one row?

`Line item ID` should identify one product line in an order. The next cell looks for sale rows that repeat a line already seen,
and what differs between the copies.

In [ ]:
with_line_id = sales[sales["Line item ID"] != 0]
repeated = with_line_id[with_line_id.duplicated(["Order ID", "Line item ID"], keep=False)]
groups = repeated.groupby(["Order ID", "Line item ID"])
what_differs = (groups.nunique() > 1).mean().mul(100).round(1).sort_values(ascending=False).head(6)
extra_rows = with_line_id.duplicated(["Order ID", "Line item ID"]).sum()
extra_value = with_line_id.loc[with_line_id.duplicated(["Order ID", "Line item ID"]), "Net sales"].sum()
print(f"Order lines recorded in more than one row: {groups.ngroups:,}, adding {extra_rows:,} extra rows")
print(f"Net sales on the extra rows: SGD {extra_value:,.0f}, {extra_value / sales['Net sales'].sum() * 100:.1f}% of kept sales")
print()
print("In those repeated lines, % of lines where each column differs between the copies:")
what_differs.to_frame("% of repeated lines")

In [ ]:
edited = groups[["Day", "Quantity ordered", "Net sales"]].nunique().gt(1).any(axis=1)
print(f"Repeated lines where day, quantity or money also differ (order edits, not stacked discounts): {edited.sum():,}")
pairs = groups["Discount name"].agg(lambda n: " + ".join(sorted(n.dropna().astype(str))))
print("Most common discount pairs on repeated lines:")
print(pairs.value_counts().head(3).to_string())
example = repeated[repeated["Discount name"].astype(str).str.contains("OPTWHS5")]["Order ID"].iloc[0]
repeated.loc[repeated["Order ID"] == example, ["Order name", "Day", "Line item ID", "Discount name", "Quantity ordered", "Net sales"]]

In [ ]:
undo_rows = orders[~SALE & (orders["Line item ID"] != 0)]
undo_repeats = undo_rows[undo_rows.duplicated(["Order ID", "Line item ID", "Refund ID"])]
print(f"Undo rows that repeat the same refund of the same line: {len(undo_repeats):,}, SGD {undo_repeats['Net sales'].sum():,.0f}")

**How to read it.** When two discounts are stacked on one item, the file writes one row per discount, and each row repeats the
item's full **Net sales**. In almost every repeated line only the discount columns differ, so adding up Net sales counts those
items twice. A few repeated lines are order edits instead. Refund rows repeat the same way, so refunds are also counted once per line.

From here on the notebook uses `lines`, the sale rows with each order line counted once.

In [ ]:
lines = sales[~((sales["Line item ID"] != 0) & sales.duplicated(["Order ID", "Line item ID"]))]
print(format(len(sales), ","), "sale rows ->", format(len(lines), ","), "rows with each order line once")

### B2. Describe, the column table

In [ ]:
profile(orders.drop(columns="channel"))

**How to read it.** Each row of this table is one column of the file. Look first at **missing %**, then compare
**different values** with the number of rows. 2 different values means a yes or no column.

Columns to be careful with:

- `Customer amount spent` and `Customer number of orders` are the customer's **lifetime** totals, repeated on every row. Adding them up counts the same customer many times.
- `Weeks since first purchase` is `-1` on first purchases. It is a code, not a number of weeks. The next cell proves it.
- `Return reason` is almost always missing, so it cannot be used.
- `Refund ID` and `Line item ID` use `0` instead of leaving the cell empty, so the missing count shows 0 even though many rows have no value. See the checks in B5.

In [ ]:
print("Weeks since first purchase = -1, compared with New or returning customer")
pd.crosstab(orders["Weeks since first purchase"] == -1, orders["New or returning customer"])

### B3. Explore, coverage over time

To count each order once, the next cell builds a small table with **one row per order**:
its first day, channel, value, number of items and whether it had a discount.

In [ ]:
per_order = lines.groupby("Order ID").agg(
    day=("Day", "min"),
    channel=("channel", "first"),
    channel_name=("Order sales channel", "first"),
    customer=("Customer ID", "first"),
    value=("Net sales", "sum"),
    items=("Quantity ordered", "sum"),
)
per_order["discounted"] = sales.assign(d=sales["Discounts"] < 0).groupby("Order ID")["d"].max()
per_order["month"] = per_order["day"].dt.to_period("M")
print(format(len(per_order), ","), "orders that were not cancelled")

orders_by_month = per_order.groupby(["month", "channel"]).size().unstack(fill_value=0)
line_chart(orders_by_month[["Website", "Marketplace", "Other"]], "Orders per month, by channel",
           ylabel="orders", colors=[GREEN, ORANGE, GREY])

all_months = pd.period_range(orders_by_month.index.min(), orders_by_month.index.max(), freq="M")
print("Months covered:", orders_by_month.index.min(), "to", orders_by_month.index.max())
print("Months with no orders at all:", len(all_months) - len(orders_by_month))
print("The first order (#1001, 24 Jul 2019) was a cancelled draft order, so the chart starts in Aug 2019.")
print("The last month stops on", orders["Day"].max().date(), "so it is not a full month.")

In [ ]:
orders_by_year = per_order.groupby([per_order["day"].dt.year, "channel"]).size().unstack(fill_value=0)
orders_by_year.index.name = "year"
orders_by_year["Website share %"] = (orders_by_year["Website"] / orders_by_year[["Website", "Marketplace", "Other"]].sum(axis=1) * 100).round(1)
orders_by_year

**How to read it.** Website orders peaked around 2022 while marketplace orders kept growing, so the website's share of
orders fell year after year. The jump in **Other** in 2024 comes from one channel, Osco, shown below.

In [ ]:
other = per_order[per_order["channel"] == "Other"]
print("Orders in the Other group, by channel name and year")
pd.crosstab(other["channel_name"], other["day"].dt.year)

### B4. Explore, distributions

In [ ]:
value_by_channel = per_order.groupby("channel")["value"].agg(["count", "median", "mean"]).round(2)
value_by_channel.columns = ["orders", "median order value", "average order value"]
print("Order value in SGD, before refunds")
value_by_channel

In [ ]:
histogram(per_order["value"], "Order value per order, SGD (log scale, $0 orders left out)", xlabel="SGD, log scale", ylabel="orders", log=True)
print("Orders worth $0:", (per_order["value"] == 0).sum())
small = per_order[(per_order["value"] > 0) & (per_order["value"] < 60)]
print("Of orders under $60, share from marketplaces:", round((small["channel"] == "Marketplace").mean() * 100, 1), "%")
print()
print("All orders together:")
numbers(per_order["value"]).to_frame("SGD")

**How to read it.** A log scale spreads out very uneven numbers, so both a $20 order and a $2,000 order are visible.
The chart has two humps because marketplace orders are much smaller than website orders (see the medians above).
The **median** is the typical order. The **average** is pulled up by a few very large orders.

In [ ]:
item_buckets = pd.cut(per_order["items"], bins=[0, 1, 2, 3, 5, 10, 10_000],
                      labels=["1", "2", "3", "4-5", "6-10", "11+"]).value_counts().sort_index()
bar_chart(item_buckets, "Items per order", xlabel="items", ylabel="orders")

In [ ]:
orders_per_id = per_order.groupby(["channel", "customer"]).size()
print("Average orders per customer ID: website", round(orders_per_id["Website"].mean(), 2),
      "| marketplace", round(orders_per_id["Marketplace"].mean(), 2))

website_ids = orders_per_id["Website"]
buckets = pd.cut(website_ids, bins=[0, 1, 2, 3, 5, 10, 100_000],
                 labels=["1", "2", "3", "4-5", "6-10", "11+"]).value_counts().sort_index()
bar_chart(buckets, "Website customers by number of orders", xlabel="orders per customer", ylabel="customers")
print("Website customers who bought only once:", round(buckets["1"] / buckets.sum() * 100, 1), "%")

**How to read it.** On marketplaces a customer ID almost never comes back (about 1 order per ID), most likely because the
platforms give the same shopper a new ID. So repeat buying can only be measured on the website.

In [ ]:
spend = per_order[per_order["channel"] == "Website"].groupby("customer")["value"].sum().sort_values(ascending=False)
top_10pct = spend.head(len(spend) // 10).sum() / spend.sum() * 100
print("Share of website sales from the top 10% of website customers:", round(top_10pct, 1), "%")
print("Spend per website customer, SGD")
numbers(spend).to_frame("SGD")

In [ ]:
print("Share of orders with any discount, by channel")
(per_order.groupby("channel")["discounted"].mean() * 100).round(1).to_frame("% of orders")

**How to read it.** An order counts as discounted if any of its rows has a negative `Discounts` value.
Discounts are common in every channel. This is the base for the case's discount-dependence question.

### B5. Check, quality

In [ ]:
print("Net sales, SGD, added up three ways")
pd.Series({
    "all rows": format(orders["Net sales"].sum(), ",.0f"),
    "sale rows only (undo rows left out)": format(orders.loc[SALE, "Net sales"].sum(), ",.0f"),
    "sale rows of orders that were not cancelled": format(sales["Net sales"].sum(), ",.0f"),
    "  ... with each order line counted once": format(lines["Net sales"].sum(), ",.0f"),
}).to_frame("Net sales")

In [ ]:
print("Rows by type: sale or undo row, and whether the order was cancelled")
pd.crosstab(SALE.map({True: "sale row", False: "undo row"}), CANCELLED.map({True: "cancelled order", False: "not cancelled"}), margins=True)

**How to read it.** Undo rows mostly carry negative money, so leaving them in makes the total **lower**, not higher.
Cancelled orders have both a sale row and an undo row, which is why the two counts overlap in the table above.
The cleaning notebooks keep sale rows of orders that were not cancelled, then subtract real refunds from their own sale.

In [ ]:
no_code = orders["Product variant SKU at time of sale"].isna()
zero_items = orders["Quantity ordered"] == 0
top_id = orders.loc[orders["Customer number of orders"].idxmax(), "Customer ID"]
checks = pd.DataFrame([
    ("Uniqueness", "Exact duplicate rows", (orders.duplicated()).sum()),
    ("Uniqueness", "Repeated keys (Order ID, Line item ID, sale or undo), Line item ID not 0",
     orders[orders["Line item ID"] != 0].duplicated(["Order ID", "Line item ID", "Order or sales reversal"]).sum()),
    ("Completeness", "Rows with no product code", no_code.sum()),
    ("Completeness", "  of those, 0 items and $0 (shipping or fee lines)", (no_code & zero_items & (orders["Net sales"] == 0)).sum()),
    ("Completeness", "  of those, 0 items but money (almost all undo rows)", (no_code & zero_items & (orders["Net sales"] != 0)).sum()),
    ("Completeness", "  of those, items but no code (all rows)", (no_code & ~zero_items).sum()),
    ("Completeness", "  kept sale lines with items but no code", (lines["Product variant SKU at time of sale"].isna() & (lines["Quantity ordered"] > 0)).sum()),
    ("Completeness", "Rows missing New or returning customer", orders["New or returning customer"].isna().sum()),
    ("Completeness", "Refund ID is 0 (0 used instead of empty)", (orders["Refund ID"] == 0).sum()),
    ("Completeness", "Line item ID is 0 (0 used instead of empty)", (orders["Line item ID"] == 0).sum()),
    ("Validity", "Sale rows with negative Net sales", (SALE & (orders["Net sales"] < 0)).sum()),
    ("Validity", "Weeks since first purchase = -1 (a code, not weeks)", (orders["Weeks since first purchase"] == -1).sum()),
    ("Validity", "Largest Customer number of orders (one customer ID)", orders["Customer number of orders"].max()),
    ("Consistency", f"Orders column added up (real orders: {orders['Order ID'].nunique():,})", orders["Orders"].sum()),
    ("Outliers", "Orders above the unusually-high line", numbers(per_order["value"]).iloc[-1]),
], columns=["dimension", "check", "result"])
checks

In [ ]:
print("The customer ID with the most orders buys only through:")
orders.loc[orders["Customer ID"] == top_id, "Order sales channel"].value_counts().to_frame("rows")

**What this tells us about Orders**

- One order spreads over about 2.7 rows, so every count of orders must use `Order ID`.
- Undo rows and cancelled orders sit next to real sales, so the total depends on which rows are kept. The team keeps sale rows of orders that were not cancelled, then subtracts real refunds.
- Most rows with no product code are shipping or fee lines. But about 4,000 kept sale lines have items and money with no code,
  mostly on marketplaces, so they cannot simply be dropped.
- Items with two stacked discounts are written twice, each copy with the full Net sales. Counting each order line once removes
  this double count.
- The customer ID with over a thousand orders is the Osco channel, not one shopper. Osco orders should be set aside in customer analysis.
- The `Orders` column does not add up to the real number of orders, so it is not used for counting.
- Order values are very uneven and differ a lot between website and marketplace, so medians by channel describe them best.
- Most website customers buy once. The top 10% of website customers bring in about a third of website sales.

### B6. More checks on Orders

Each check below looks at one thing a normal column table would miss.

**Does Net sales mean the same thing in every month?** Singapore prices include GST (tax). Normally `Total sales` has GST and `Net sales` does not.

In [ ]:
item_rows = lines[(lines["Quantity ordered"] > 0) & (lines["Net sales"] > 0)].copy()
item_rows["GST shown separately"] = item_rows["Total sales"] > item_rows["Net sales"] * 1.05
recent = item_rows[item_rows["Day"] >= "2025-06-01"]
(recent.groupby([recent["Day"].dt.to_period("M"), "channel"])["GST shown separately"].mean().unstack()[["Website", "Marketplace"]] * 100).round(0)

In [ ]:
marketplace_items = item_rows[item_rows["channel"] == "Marketplace"]
top_code = marketplace_items["Product variant SKU at time of sale"].value_counts().index[0]
same_item = marketplace_items[(marketplace_items["Product variant SKU at time of sale"] == top_code) & (marketplace_items["Day"] >= "2025-01-01")]
price = (same_item["Net sales"] - same_item["Discounts"]) / same_item["Quantity ordered"]
switch = same_item["Day"] >= "2025-08-26"
print("Most sold marketplace item:", top_code)
print(f"Net sales per unit before discount: {price[~switch].median():.2f} before 26 Aug 2025, {price[switch].median():.2f} after "
      f"(x{price[switch].median() / price[~switch].median():.2f})")
share = marketplace_items.loc[marketplace_items["Day"] >= "2025-08-26", "Net sales"].sum() / marketplace_items["Net sales"].sum() * 100
print(f"Marketplace Net sales from 26 Aug 2025 onward: {share:.1f}% of all marketplace Net sales")

**How to read it.** Up to late August 2025, marketplace rows show GST on top of Net sales. After that, Total equals Net and the
same item's Net price jumps by 9%, the GST rate. So from **26 Aug 2025 marketplace Net sales include GST**, which makes recent
marketplace sales look about 9% higher than they are. Cleaning divides those rows by 1.09.

**When do the discount detail columns start?**

In [ ]:
detail = ["Discount code", "Discount name", "Discount title", "Discount value"]
pd.DataFrame({
    "first day filled": [orders.loc[orders[c].notna(), "Day"].min().date() for c in detail],
    "discounted sale rows before that day": [(SALE & (orders["Discounts"] < 0) & (orders["Day"] < orders.loc[orders[c].notna(), "Day"].min())).sum() for c in detail],
}, index=detail)

**How to read it.** All four discount detail columns start on exactly 1 Jan 2023, in every channel, although many earlier rows
were discounted. This is a cut-off in the export, so *which* discount was used is only known from 2023.

**What kind of discounts do marketplace orders get?**

In [ ]:
mp = lines[(lines["channel"] == "Marketplace") & (lines["Discounts"] < 0)].copy()
name = mp["Discount name"].astype(str)
mp["price match"] = name.str.contains("price match|match shopee price", case=False)
mp["platform funded"] = name.str.contains("lazada voucher|shopee coins", case=False)
mp["no name (before 2023)"] = mp["Discount name"].isna()
print("Discounted marketplace lines:", format(len(mp), ","))
mp[["price match", "platform funded", "no name (before 2023)"]].sum().to_frame("lines")

In [ ]:
seller = mp.loc[~mp["price match"] & ~mp["platform funded"], "Order ID"].unique()
mp_orders = per_order[per_order["channel"] == "Marketplace"]
print(f"Marketplace orders with any discount: {mp_orders['discounted'].mean() * 100:.1f}%")
print(f"... counting only discounts that are not price match or platform funded: {mp_orders.index.isin(seller).mean() * 100:.1f}%")

**How to read it.** Many marketplace "discounts" are price matching or vouchers paid by the platform, not Weavve promotions.
Discount dependence on marketplaces should be shown both ways.

**Do the lifetime columns match the file?**

In [ ]:
all_orders = orders.groupby("Customer ID")["Order ID"].nunique()
kept_orders = per_order.groupby("customer").size()
shopify_count = orders.groupby("Customer ID")["Customer number of orders"].max()
total_once = orders[SALE].drop_duplicates(["Order ID", "Line item ID", "Discount name"]).groupby("Customer ID")["Total sales"].sum()
pd.Series({
    "Customer number of orders = all orders, cancelled included (%)": round((shopify_count == all_orders).mean() * 100, 1),
    "Customer number of orders = orders not cancelled (%)": round((shopify_count == kept_orders.reindex(shopify_count.index).fillna(0)).mean() * 100, 1),
    "Customer amount spent = Total sales, within $1 (%)": round(((orders.groupby("Customer ID")["Customer amount spent"].max() - total_once.reindex(all_orders.index).fillna(0)).abs() < 1).mean() * 100, 1),
    "customer IDs with only cancelled orders": int((kept_orders.reindex(all_orders.index).fillna(0) == 0).sum()),
}).to_frame("value")

**How to read it.** The lifetime columns count cancelled orders too, and `Customer amount spent` is on the Total sales basis,
with GST and shipping. So they cannot be compared with figures built from Net sales. Customer measures are rebuilt from kept orders.

**Other checks**

In [ ]:
text_codes = orders["Discount code"].astype(str)
osco = per_order[per_order["channel_name"] == "Osco"]
website_days = per_order[(per_order["channel"] == "Website") & (per_order["day"] >= "2022-01-01")]["day"].dt.normalize()
all_days = pd.Series(pd.date_range(website_days.min(), website_days.max(), freq="D"))
no_website = all_days[~all_days.isin(set(website_days))]
runs = no_website.diff().dt.days.ne(1).cumsum()
longest = no_website.groupby(runs).agg(["min", "max", "size"]).sort_values("size").iloc[-1]
cancelled_net = orders[CANCELLED].groupby("Order ID")["Net sales"].sum()
whs = per_order.index.isin(lines.loc[lines["Discount code"].astype(str).str.upper().str.startswith("WHS"), "Order ID"])
pd.DataFrame([
    ("Discount codes that Excel turned into dates", orders.loc[text_codes.str.match(r"^\d{4}-\d{2}-\d{2}"), "Order ID"].nunique(), "orders; the real code is lost"),
    ("Osco customer IDs", osco["customer"].nunique(), f"one ID holds {osco['customer'].value_counts().iloc[0]:,} orders, later orders get a new ID each"),
    ("Orders with 20 or more items", int((per_order["items"] >= 20).sum()), f"SGD {per_order.loc[per_order['items'] >= 20, 'value'].sum():,.0f}, bulk buyers"),
    ("Longest run of days with no website orders (since 2022)", int(longest["size"]), f"{longest['min']:%d %b %Y} to {longest['max']:%d %b %Y}"),
    ("Cancelled orders that do not net to $0", int((cancelled_net.abs() > 1).sum()), f"of {len(cancelled_net):,} cancelled orders"),
    ("Warehouse sale orders (WHS codes), by channel", int(whs.sum()), ", ".join(f"{k} {v}" for k, v in per_order[whs]["channel_name"].value_counts().items())),
], columns=["check", "result", "note"])

**How to read it.** Each line is one extra check. Warehouse sale events were sold through different channels over time
(website, then Draft Orders), which moves some real shoppers into "Other". Bulk orders and the Osco channel should be kept apart
from normal shoppers.

## C. File 4, Products

**Each row shows** one product in one size or colour, known as a SKU.

In [ ]:
print(len(products), "rows,", products.shape[1], "columns")
print("Different product codes (Code):", products["Code"].nunique(), "| repeated codes:", products["Code"].duplicated().sum())
print("Different products (Product Name):", products["Product Name"].nunique())
profile(products)

In [ ]:
empty = [c for c in products.columns if products[c].isna().all()]
same_everywhere = [c for c in products.columns if products[c].notna().all() and products[c].nunique() == 1]
flags = [c for c in products.columns if products[c].nunique() == 1 and products[c].isna().any() and c not in empty]
print("Empty columns:", empty)
print("Same value on every row:", same_everywhere)
print("Columns that mark only some rows (flags):", {c: products[c].dropna().iloc[0] for c in flags})
print("Rows where Brand is 0 instead of a brand name:", (products["Brand"].astype(str) == "0").sum())

In [ ]:
bar_chart(products["Category"].value_counts(), "Product codes by category", ylabel="product codes", sideways=True)

### Check, do the products sold appear in this list?

In [ ]:
real_items = sales[sales["Quantity ordered"] > 0]
code_col = "Product variant SKU at time of sale"
has_code = real_items[code_col].notna()
in_list = real_items[code_col].isin(set(products["Code"]))
join = pd.DataFrame({
    "rows": [in_list.sum(), (has_code & ~in_list).sum(), (~has_code).sum()],
    "sales value": [real_items.loc[in_list, "Net sales"].sum(), real_items.loc[has_code & ~in_list, "Net sales"].sum(),
                    real_items.loc[~has_code, "Net sales"].sum()],
}, index=["code found in the product list", "code NOT in the product list", "no code at all"])
join["% of rows"] = (join["rows"] / join["rows"].sum() * 100).round(1)
join["% of value"] = (join["sales value"] / join["sales value"].sum() * 100).round(1)
join["sales value"] = join["sales value"].round(0)
join

In [ ]:
missing = real_items[has_code & ~in_list]
last_sold = missing.groupby(code_col)["Day"].max().dt.year
print(missing[code_col].nunique(), "codes sold but not in the list. The year each was last sold:")
last_sold.value_counts().sort_index().to_frame("codes")

### More checks on Products

In [ ]:
text_columns = ["Sub Category", "Product Name", "Option 1", "Option 2"]
spaces = {c: int((products[c].dropna().astype(str) != products[c].dropna().astype(str).str.strip()).sum()) for c in text_columns}
print("Values with extra spaces at the start or end:", spaces)
print("Code schemes:", products["Code"].str[:3].value_counts().head(3).to_dict())
print("Brand = 0, by code scheme:", products.loc[products["Brand"].astype(str) == "0", "Code"].str[:3].value_counts().to_dict())
imperfect_name = products["Product Name"].str.contains("imperfect", case=False)
print("Named Imperfect but not flagged in Product Sub Type:", int((imperfect_name & products["Product Sub Type"].isna()).sum()))
print()
print("Best-selling items in 'Accessories & Packaging' (order rows):")
code_to_category = products.set_index("Code")["Category"]
sold_lines = lines[lines["Quantity ordered"] > 0].assign(category=lines["Product variant SKU at time of sale"].map(code_to_category))
sold_lines.loc[sold_lines["category"] == "Accessories & Packaging", "Product variant SKU at time of sale"].map(
    products.set_index("Code")["Product Name"]).value_counts().head(5).to_frame("order rows")

**How to read it.** Some text values carry stray spaces, so the same size or colour can look like two different values until they
are trimmed. Products use two code schemes, an older *BR-* and a newer *WH-*, and the *Brand = 0* rows are mostly the new scheme.
The "Accessories & Packaging" category holds real merchandise like the silk eye mask, so category totals need a check before use.

**What this tells us about Products**

- `Code` is unique, so it is the right column to join with Orders. Brand and some categories are unreliable and text needs trimming.
- Several columns are empty or identical on every row, so they carry no information. `Product Sub Type` is not empty,
  it flags a few items as *Imperfect*.
- Some sold items cannot be matched to the list. Most of those codes were last sold years ago, which matches the
  project's warning that past products no longer appear in the current catalogue.

## D. File 8, Discounts

**Each row shows** one discount entry. The file has 5 tabs. The main tab lists every entry. The other 4 are Weavve's own summaries.

In [ ]:
discount_book = pd.ExcelFile(paths.path("discounts"))
tab_sizes = []
for name in discount_book.sheet_names:
    sheet = discount_book.parse(name)
    tab_sizes.append((name.strip(), sheet.shape[0], sheet.shape[1]))
pd.DataFrame(tab_sizes, columns=["tab", "rows", "columns"])

In [ ]:
profile(discounts)

In [ ]:
names = discounts["Name"].astype(str)
print("Rows:", format(len(discounts), ","), "| different names:", format(names.nunique(), ","),
      "| rows whose name appears more than once:", format(names.duplicated(keep=False).sum(), ","))
print("Entries with TEST in the name:", names.str.contains("test", case=False).sum())
start_all = pd.to_datetime(discounts["Start"].astype(str).str[:19], errors="coerce")
print("Report taken on or after:", start_all.max(), "(the latest start). Orders run to", orders["Day"].max().date())
print("Rows used more times than their usage limit:", int((discounts["Times Used In Total"] > discounts["Usage Limit Per Code"]).sum()),
      "(these rows hold many generated codes under one name)")

**Families.** Most entries are copies of a few campaigns. The next cell groups them by a pattern in the name:

| Family | How it is spotted |
|---|---|
| Email code (Klaviyo) | the name ends with *Custom Ecommerce Email* |
| Pop-up code (Alia) | the name starts with *[Alia* |
| Loyalty reward | the name starts with *WHLP* |
| Loyalty welcome and birthday | the name starts with *loyaltywelcome* or *BDAY* |
| Welcome code | the name starts with *welcome* |
| Chat code (Tidio) | the name contains *tidio* |
| Other per-customer campaign | *Fulfilled20*, *CART10*, *VisitedNotBought*, *organic_search* |
| Other | anything else, which includes the evergreen campaign codes |

In [ ]:
family = pd.Series("Other", index=discounts.index)
patterns = [("Email code (Klaviyo)", "Custom Ecommerce Email"), ("Pop-up code (Alia)", r"^\[Alia"),
            ("Loyalty reward", "^WHLP"), ("Loyalty welcome and birthday", "(?i)^(?:loyaltywelcome|bday)"),
            ("Welcome code", "(?i)^welcome"), ("Chat code (Tidio)", "(?i)tidio"),
            ("Other per-customer campaign", "(?i)^(?:fulfilled20|cart10|visitednotbought|organic_search)")]
for label, pattern in patterns:
    family[names.str.contains(pattern) & (family == "Other")] = label

used = discounts["Times Used In Total"] > 0
families = pd.DataFrame({"entries": family.value_counts(), "ever used": used.groupby(family).sum()})
families["used %"] = (families["ever used"] / families["entries"] * 100).round(1)
families["one use per customer %"] = (discounts["Applies Once Per Customer"].eq(1).groupby(family).mean() * 100).round(1)
families.sort_values("entries", ascending=False)

In [ ]:
print("Evergreen codes: entries used more than 100 times")
discounts.loc[discounts["Times Used In Total"] > 100, ["Name", "Value", "Value Type", "Customer Selection", "Times Used In Total"]].sort_values(
    "Times Used In Total", ascending=False)

**How to read it.** These few codes are the **evergreen templates**, nearly all open to every customer (`all`) and used many times. The families
above are the **per-customer copies**, each meant for one person. The project instructions warn about exactly this mix.

In [ ]:
usage = pd.cut(discounts["Times Used In Total"], bins=[-1, 0, 1, 10, 100, 100_000],
               labels=["never", "once", "2-10", "11-100", "over 100"]).value_counts().sort_index().to_frame("entries")
usage["%"] = (usage["entries"] / usage["entries"].sum() * 100).round(1)
print("How many times each entry was used")
usage

In [ ]:
start = pd.to_datetime(discounts["Start"].astype(str).str[:10], errors="coerce")
bar_chart(start.dt.year.value_counts().sort_index(), "Discount entries by start year (2019 and 2026 are part years)",
          ylabel="entries")
print("Start dates that could not be read:", start.isna().sum())

### Check, when do Orders record discount codes?

In [ ]:
website_orders = sales[sales["channel"] == "Website"].groupby("Order ID").agg(
    year=("Day", lambda d: d.min().year),
    has_code=("Discount code", lambda c: c.notna().any()),
    discounted=("Discounts", lambda d: (d < 0).any()),
)
coverage_by_year = (website_orders.groupby("year")[["has_code", "discounted"]].mean() * 100).round(1)
coverage_by_year.columns = ["% of website orders with a code", "% of website orders with a discount"]
coverage_by_year

**How to read it.** Before 2023 website orders were often discounted, but **no discount code was recorded**.
Codes only appear in Orders from 2023. So any analysis of discounts by code only works from 2023. For earlier years,
the `Discounts` amount is the only record.

### Check, do discount codes used in Orders appear in this file?

In [ ]:
# The Name column is not always a clean code. Clean it before matching:
#   "2UMO-JNOV-IC6T - Custom Ecommerce Email"  ->  "2UMO-JNOV-IC6T"   (keep the part before " - ")
#   "welcome10 (2022-08-15 09:37)"             ->  "WELCOME10"        (drop the date in brackets)
clean_code = names.str.split(" - ").str[0]
clean_code = clean_code.str.replace(r"\s*\(\d{4}-\d{2}-\d{2}[^)]*\)\s*$", "", regex=True).str.strip().str.upper()

print("Rows with a discount code in Orders, by channel:")
print(sales.groupby("channel")["Discount code"].count().to_string())
print()
used_codes = sales.loc[sales["channel"] == "Website", "Discount code"].dropna().str.upper()
for label, known in [("names as they are", set(names.str.upper())), ("names cleaned", set(clean_code))]:
    found = used_codes.isin(known)
    print(f"Website codes found, {label}: {found.mean() * 100:.1f}% of rows, "
          f"{pd.Series(used_codes.unique()).isin(known).mean() * 100:.1f}% of different codes")
print()
used_entries = clean_code[discounts["Times Used In Total"] > 0]
all_order_codes = set(sales["Discount code"].dropna().str.upper())
print(f"From the report side: {used_entries.isin(all_order_codes).mean() * 100:.1f}% of the {len(used_entries):,} used entries appear in Orders")
print()
missing_rows = used_codes[~used_codes.isin(set(clean_code))]
print("Unmatched website code rows by the first 3 letters of the code:")
print(missing_rows.str[:3].value_counts().head(4).to_string())
print()
print("Most used website codes that are still not found by name:")
used_codes[~used_codes.isin(set(clean_code))].value_counts().head(5).to_frame("order rows")

**Why so many codes are "not found".** Many report rows are named by campaign title, like *loyaltywelcome10 (2024-04-03 12:02)*,
and each such row stands for many generated codes. The next cell pairs each code prefix with its campaign and checks that the
number of orders using the prefix equals the campaign's *Times Used* in the report (orders up to the report date).

In [ ]:
stem = names.str.replace(r"\s*\(\d{4}-\d{2}-\d{2}[^)]*\)\s*$", "", regex=True).str.split(" - ").str[0].str.strip().str.lower()
report_uses = discounts.groupby(stem)["Times Used In Total"].sum()
prefix_to_campaign = [("LWC10", ["loyaltywelcome10", "loyaltywelcome10reissue"]), ("VNBNOI10", ["visitednotboughtbeforenewoptin10"]),
                      ("WC10SIN", ["welcome10signupincomplete"]), ("WC8ALF", ["welcome8aliafirst"]), ("WC10ALF", ["welcome10aliafirstreissue"]),
                      ("WCSTW", ["welcomestw10issue"]), ("WCS", ["welcome10"]), ("FUL20", ["fulfilled20discount"]),
                      ("LCP40", ["loyalty40off"]), ("TDWC8", ["tidiowelcome8"]), ("OSB", ["organic_search_bedsheets"]), ("11-11SALE", ["1111ext"])]
upto_report = orders[(orders["Day"] <= start_all.max()) & orders["Discount code"].notna()]
upto_codes = upto_report["Discount code"].str.upper()
check = []
for prefix, campaigns in prefix_to_campaign:
    check.append((prefix, ", ".join(campaigns), upto_report.loc[upto_codes.str.startswith(prefix), "Order ID"].nunique(),
                  int(sum(report_uses.get(c, 0) for c in campaigns))))
pd.DataFrame(check, columns=["code prefix", "campaign in the report", "orders using it", "times used in the report"])

In [ ]:
by_family = used_codes.str.startswith(tuple(p for p, _ in prefix_to_campaign)) | used_codes.str.match(r"^AL[A-Z0-9]{8}$")
explained = used_codes.isin(set(clean_code)) | by_family
print(f"Website code rows explained by name or by campaign prefix (AL + 8 characters are Alia pop-up codes): {explained.mean() * 100:.1f}%")
print(f"Still unexplained: {(~explained).sum():,} rows, {(~explained).mean() * 100:.1f}%")

**What this tells us about Discounts**

- The file is not one row per code. Most entries are per-customer copies of a few campaigns, as the project instructions warn,
  so analysis should group them into families.
- Most entries were never used, so counting codes says little about how often discounts are used. The Orders file shows that.
- `Start` and `End` are stored as text, so they are turned into dates before use.
- Marketplace orders carry no discount codes at all. Their discounts only show as names like *Shopee Price Match*.
- Orders only record discount codes from 2023, so most report entries, which are older, can never be matched. Discounts by code
  can only be studied from 2023. For earlier years, use the `Discounts` amount.
- Matching codes by name finds only two thirds, because many report rows are named by campaign and hold many generated codes.
  Matching by code prefix, the order counts equal the report's *Times Used* for most campaigns, and only a small share of code use
  stays unexplained. *11-11SALE* is the code the report lists as *1111EXT*, renamed after use.
- The report is a snapshot from early August 2026, while Orders run to late August, so comparisons should stop at the report date.

## E. File 9.1, Loyalty

**Each row shows** one loyalty voucher. `source` says how the customer got it, for example by spending points or automatically.
The file name mentions customers and earning actions, but it only has the *Redemptions* tab.

In [ ]:
print(len(loyalty), "rows,", loyalty.shape[1], "columns | from", loyalty["date"].min().date(), "to", loyalty["date"].max().date())
profile(loyalty)

In [ ]:
vouchers_per_month = loyalty.groupby(loyalty["date"].dt.to_period("M")).size()
line_chart(vouchers_per_month, "Loyalty vouchers issued per month (the first month has only a few days)", ylabel="vouchers",
           colors=GREEN)

In [ ]:
by_source = loyalty.groupby("source").agg(vouchers=("discount_code", "count"), used=("times_used", "sum"))
by_source["used %"] = (by_source["used"] / by_source["vouchers"] * 100).round(1)
by_source.sort_values("vouchers", ascending=False)

In [ ]:
loyalty["how obtained"] = np.where(loyalty["points"] < 0, "spent points", "no points spent")
by_points = loyalty.groupby("how obtained").agg(vouchers=("discount_code", "count"), used=("times_used", "sum"))
by_points["used %"] = (by_points["used"] / by_points["vouchers"] * 100).round(1)
by_points

In [ ]:
by_reward = loyalty.groupby("description").agg(vouchers=("discount_code", "count"), used=("times_used", "sum"))
by_reward["used %"] = (by_reward["used"] / by_reward["vouchers"] * 100).round(1)
by_reward.sort_values("vouchers", ascending=False)

In [ ]:
report_value = discounts.assign(code=names.str.upper()).drop_duplicates("code").set_index("code")["Value"]
loyalty["value in report"] = loyalty["discount_code"].str.upper().map(report_value)
bought = loyalty[(loyalty["points"] < 0) & loyalty["description"].isin(["$90 off", "$50 off"])]
period = pd.Series(np.where(bought["date"] < "2025-03-01", "1 before Mar 2025", "2 from Mar 2025"), index=bought.index, name="period")
print("Vouchers bought with points: the label against the value the code really gave in the discount report (SGD)")
pd.crosstab([period, bought["description"].rename("label"), bought["points"].rename("points")],
            (-bought["value in report"]).rename("value really given"))

**How to read it.** Before Mar 2025 every voucher bought with points shows the opposite value: 600 points bought a voucher
labelled *\$50 off* whose code gave \$90 (a few \$70), and 1,000 points bought one labelled *\$90 off* whose code gave \$50 (a few \$40).
From Mar 2025 label and code agree. Referral vouchers were not affected. So a reward's value must come from the discount report, not from `description`.

In [ ]:
with_order = loyalty["order_id"].dropna().astype("int64")
print("Vouchers used on an order:", len(with_order), "of", len(loyalty))
print("Of those, found in the Orders file:", round(with_order.isin(set(orders["Order ID"])).mean() * 100, 1), "%")

**How to read it.** `used %` is the share of vouchers that were used on an order.

**What this tells us about Loyalty**

- Name and email are empty, so a voucher links to a customer only through `order_id`, and every used voucher does match an order.
- Vouchers bought by spending points are used far more often than vouchers given without points. The automatic *15% off*
  vouchers from *Earning Rules* are almost never used, while checkout vouchers are always used, likely because they are created at the moment of checkout.
  Any measure of loyalty engagement has to look at these groups separately.
- `points` is negative when points were spent and 0 otherwise. `amount_used_cents` is 0 on every row, so it carries no information.
- Before Mar 2025, vouchers bought with points gave the other reward's value (\$50 label gave \$90 and back), so the value given must be taken from the discount report.

## F. File 12, Google Ads

**Each row shows** one keyword in one campaign. The file has 4 tabs.

Two terms: **impressions** is how often an ad was shown, and **click-through rate (CTR)** is clicks divided by impressions.

In [ ]:
ads_book = pd.ExcelFile(paths.path("ads"))
tab_sizes = []
for name in ads_book.sheet_names:
    sheet = ads_book.parse(name)
    tab_sizes.append((name.strip(), sheet.shape[0], sheet.shape[1]))
pd.DataFrame(tab_sizes, columns=["tab", "rows", "columns"])

In [ ]:
profile(ads)

In [ ]:
print("Keyword status today:")
print(ads["Status"].value_counts().to_string())
print()
pd.Series({
    "keyword rows": format(len(ads), ","),
    "different keywords (cleaned)": format(ads["kw"].nunique(), ","),
    "rows that ever showed an ad": format(int((ads["Impr."] > 0).sum()), ","),
    "total cost (SGD)": format(ads["Cost"].sum(), ",.0f"),
    "total clicks": format(int(ads["Clicks"].sum()), ","),
    "click-through rate, all rows (%)": round(ads["Clicks"].sum() / ads["Impr."].sum() * 100, 2),
    "conversions": format(ads["Conversions"].sum(), ",.0f"),
}).to_frame("value")

In [ ]:
by_keyword = ads.groupby("kw")[["Cost", "Clicks", "Impr.", "Conversions"]].sum().sort_values("Cost", ascending=False)
top_share = by_keyword["Cost"].head(len(by_keyword) // 10).sum() / by_keyword["Cost"].sum() * 100
print("The top 10% of keywords took", round(top_share, 1), "% of all spend")
by_keyword["CTR %"] = (by_keyword["Clicks"] / by_keyword["Impr."] * 100).round(1)
by_keyword.head(10).round({"Cost": 0, "Conversions": 0})

**Weavve's own tiers.** The `Filtered` tab repeats the keywords and tags each with a tier from Weavve's own keyword list.
The `Categories` tab is the legend. Keywords Weavve marked **red** (do not retarget) are shown by cell colour in Excel,
so the helper `load.ads_red_list()` reads the colour.

In [ ]:
filtered = ads_book.parse("Filtered")
legend = ads_book.parse("Categories", header=None).iloc[:, 1].dropna().tolist()
print("Tier legend (Categories tab):")
for line in legend:
    print(" -", line)
print()
print("Keywords marked red, do not retarget:", len(load.ads_red_list()))
filtered["KW List tier colour"].fillna("(no tier)").value_counts().to_frame("keyword rows")

In [ ]:
negatives = ads_book.parse("Negative Keyword report ")
words = negatives[negatives["Keyword or list"] == "Keyword"]
distinct_words = words["Negative keyword"].str.strip('[]"+').str.lower().nunique()
print(f"{len(negatives):,} rows: {len(words):,} blocked-word rows holding {distinct_words:,} different words, "
      f"plus {(negatives['Keyword or list'] == 'List').sum()} references to shared lists whose contents are not in the file")
words["Match type"].value_counts().to_frame("blocked-word rows")

### More checks on Google Ads

In [ ]:
by_campaign = ads.groupby("Campaign")[["Clicks", "Conversions", "Conv. value", "Cost"]].sum()
by_campaign["value per conversion"] = (by_campaign["Conv. value"] / by_campaign["Conversions"]).round(2)
print(f"All conversions: {ads['Conversions'].sum():,.0f}, worth SGD {ads['Conv. value'].sum() / ads['Conversions'].sum():.2f} each on average. The two campaigns below hold "
      f"{by_campaign.loc[['Bed Linen #2', 'Competitor (old)'], 'Conversions'].sum() / ads['Conversions'].sum() * 100:.0f}% of them.")
by_campaign.loc[["Bed Linen #2", "Competitor (old)"]].round({"Conversions": 0, "Conv. value": 0, "Cost": 0})

**How to read it.** These two campaigns report many conversions worth almost nothing, so they were counting something other
than sales. Keywords should be judged on conversion **value**, not conversion counts.

In [ ]:
red_words = set(load.ads_red_list())
clean_kw = ads["Keyword"].astype(str).str.strip('[]"+').str.replace("+", "", regex=False).str.lower()
is_red_word = clean_kw.isin(red_words)
brand = ads["Campaign"].str.contains("brand", case=False)
only_in_negatives = set(negatives["Campaign"].dropna()) - set(ads["Campaign"])
pd.Series({
    "spend on keywords whose word is marked red (SGD)": format(ads.loc[is_red_word, "Cost"].sum(), ",.0f"),
    "share of all spend (%)": round(ads.loc[is_red_word, "Cost"].sum() / ads["Cost"].sum() * 100, 1),
    "spend in brand campaigns (SGD)": format(ads.loc[brand, "Cost"].sum(), ",.0f"),
    "campaigns found only in the negatives tab": len(only_in_negatives),
}).to_frame("value")

**How to read it.** Red marks are cell colours on some rows, so the same word can appear uncoloured elsewhere. Matching by the word
itself shows how much was spent on terms Weavve now treats as do-not-retarget. Some campaigns (Shopping, Performance Max and
other countries) appear only in the negatives tab, so the spend here covers search keywords only, not all of Google Ads.

**What this tells us about Google Ads**

- This is past history. Every keyword is paused or not eligible now. The Filtered tab repeats the keyword report exactly and adds Weavve's tiers.
- Spend was very concentrated. The brand word `weavve` was one of the most expensive keywords.
- About half of all recorded conversions come from two campaigns whose conversions carry almost no value, so conversion counts overstate results.
- The spend figures cover search keywords only, not Shopping or Performance Max campaigns.
- The `Filtered` and `Categories` tabs, the red marks and the blocked words together show which terms Weavve itself treated as
  off limits. These are inputs to the Track B keyword work.
- Some columns need cleaning before use. `Max. CPC` and `Interaction rate` are text, and `Final URL` is empty.

## G. File 2, Semrush

**Each row shows** one keyword result in one month. There is one CSV file per month, stacked into one table.
Semrush figures are **estimates**, not Weavve's own traffic. A keyword can appear more than once in a month: with several Weavve
pages ranking for it, or again in an image or *People also ask* box (`Position Type` says which). So keywords are counted once,
while estimated visits add up across pages.

In [ ]:
print("Months:", semrush["month"].min().date(), "to", semrush["month"].max().date(), "|", semrush["month"].nunique(), "files")
every_month = pd.date_range(semrush["month"].min(), semrush["month"].max(), freq="MS")
print("Months missing:", len(every_month) - semrush["month"].nunique())
print("Rows where the same keyword appears again in the same month:", format(semrush.duplicated(["month", "Keyword"]).sum(), ","))
organic_rows = semrush[semrush["Position Type"] == "Organic"]
print("  of which within organic results (another Weavve page for the same keyword):",
      format(organic_rows.duplicated(["month", "Keyword"]).sum(), ","))
print()
print(semrush["Position Type"].value_counts().to_string())
profile(semrush.drop(columns=["month", "source_file"]))

In [ ]:
semrush["timestamp month"] = pd.to_datetime(semrush["Timestamp"], errors="coerce").dt.to_period("M").dt.to_timestamp()
semrush["carried over"] = semrush["timestamp month"] < semrush["month"]
carried = semrush.groupby("month")["carried over"].mean().mul(100).round(0)
print("Share of rows whose Timestamp is from an earlier month than the file (carried over), recent months (%):")
print(carried[carried.index >= "2025-06-01"].to_string())

**How to read it.** From September 2025, each monthly file repeats many rows from earlier months, with their old dates. Those rows
are not fresh rankings, so counts are shown both with and without them.

In [ ]:
organic = semrush[semrush["Position Type"] == "Organic"]
fresh = organic[~organic["carried over"]]
semrush_month = pd.DataFrame({
    "rows in the file": semrush.groupby("month").size(),
    "organic keywords": organic.groupby("month")["Keyword"].nunique(),
    "organic keywords in top 10": organic[organic["Position"] <= 10].groupby("month")["Keyword"].nunique(),
    "top 10, fresh rows only": fresh[fresh["Position"] <= 10].groupby("month")["Keyword"].nunique(),
    "estimated visits, all result types": semrush.groupby("month")["Traffic"].sum(),
    "estimated visits": organic.groupby("month")["Traffic"].sum(),
    "deepest position": semrush.groupby("month")["Position"].max(),
})
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
semrush_month[["organic keywords in top 10", "top 10, fresh rows only"]].plot(ax=axes[0], color=[GREY, GREEN])
tidy(axes[0], "Keywords in Google's top 10", ylabel="keywords")
semrush_month[["estimated visits, all result types", "estimated visits"]].plot(ax=axes[1], color=[GREY, ORANGE])
axes[1].legend(["all result types", "organic results only"])
tidy(axes[1], "Estimated monthly visits", ylabel="visits")
plt.tight_layout()
plt.show()

**How to read it.** Counted on all rows, top-10 keywords seem to climb into early 2026. Counted on fresh rows only, much of that
climb disappears, so it came from carried-over rows. Estimated organic visits peaked in mid-2024 and fell in 2026. Part of that fall
is Semrush moving visits into AI overview and *People also ask* results, which is why the grey line falls less. Every monthly file
goes down to about position 100, and the files still grow in size over time (below).

In [ ]:
print("Deepest position in each monthly file: from", semrush_month["deepest position"].min(), "to", semrush_month["deepest position"].max())
numbers(semrush_month["rows in the file"]).to_frame("rows per monthly file")

In [ ]:
print("Brand keywords (containing 'weavv') in Semrush, all months:", semrush.loc[semrush["Keyword"].str.contains("weavv", case=False), "Keyword"].nunique())
latest = fresh[fresh["month"] == fresh["month"].max()].sort_values("Position").drop_duplicates("Keyword")
position_buckets = pd.cut(latest["Position"], bins=[0, 3, 10, 20, 50, 100],
                          labels=["1-3", "4-10", "11-20", "21-50", "51-100"]).value_counts().sort_index()
bar_chart(position_buckets, "Google position of Weavve's organic keywords, latest month (fresh rows)", xlabel="position", ylabel="keywords")
latest.nlargest(10, "Traffic")[["Keyword", "Position", "Search Volume", "Traffic"]]

**What this tells us about Semrush**

- All 43 months are there, with no gaps, and very few missing values.
- The same keyword can appear several times in a month, mostly because several Weavve pages rank for it. Keywords must be counted once.
- From September 2025 the files repeat old rows, which inflates recent keyword counts. Fresh rows show a much flatter picture.
- Estimated organic visits fell in 2026, partly because Semrush now counts some visits under AI overview and similar results.
- Brand searches barely appear in Semrush, so it cannot be used to judge brand terms. Search Console can.
- The monthly files grow over time, so counts of keywords should be compared with that in mind.

## H. File 1, Search Console

**What it is.** A folder of 38 Excel workbooks exported from Google Search Console. Each workbook compares two months
side by side and has the same 6 main tabs. These are Weavve's **real** Google search clicks. The team's figures are for Singapore.

In [ ]:
first = pd.ExcelFile(books[0])
print(len(books), "workbooks,", lockfiles, "Excel lock files skipped (temporary files Excel leaves behind, not data)")
print("Tabs in the first workbook:", first.sheet_names)

### Read every workbook once

This loop opens each workbook one time and notes: the columns in each tab, the workbook's **main month** (the later of the
two months it compares), the date and country written in its Filters tab, the Singapore clicks for both months, and how many
clicks its Queries tab holds. It takes a little while.

In [ ]:
def month_of(header):
    """Read the month from a column header like '5/1/23 - 5/31/23 Clicks'."""
    found = re.match(r"(\d+)/1/(\d+)", str(header).strip())
    if found:
        return pd.Timestamp(year=2000 + int(found.group(2)) % 100, month=int(found.group(1)), day=1)
    return None


def metric_column(columns, month, word):
    """Find a month's column for a measure (Clicks or Imp). Headers come in styles like '5/1/23 - 5/31/23 Clicks' or 'Jul Clicks'."""
    for c in columns:
        text = str(c).strip()
        if word in text and not text.startswith("Diff"):
            if month_of(text) == month or (month_of(text) is None and text.startswith(month.strftime("%b"))):
                return c
    return None


def clicks_column(columns, month):
    return metric_column(columns, month, "Clicks")


layout, workbooks, gsc_rows = [], [], []
for book in books:
    workbook = pd.ExcelFile(book)
    for tab in workbook.sheet_names:
        layout.append((tab, workbook.parse(tab, nrows=0).shape[1]))

    countries = workbook.parse("Countries")
    months = [month_of(c) for c in countries.columns if month_of(c) is not None]
    main_month = max(months)                  # the later of the two months the workbook compares

    filters = workbook.parse("Filters", header=None)
    date_text = filters.loc[filters[0] == "Date", 1].iloc[0]             # the first Date row
    filters_month = pd.to_datetime(str(date_text).split("-")[0].strip(), format="%b %d, %Y")
    country_rows = filters.loc[filters[0] == "Country", 1]
    country_filter = country_rows.iloc[0] if len(country_rows) else "none"

    singapore = countries[countries["Country"] == "Singapore"].iloc[0]
    for column in countries.columns:
        if str(column).strip().endswith("Clicks") and month_of(column) is not None:
            gsc_rows.append((Path(book).name, month_of(column), int(singapore[column]), month_of(column) == main_month))

    queries = workbook.parse("Queries")
    q_column = clicks_column(queries.columns, main_month)
    query_clicks = pd.to_numeric(queries[q_column], errors="coerce").sum()
    pages = workbook.parse("Pages")
    page_clicks = pd.to_numeric(pages[clicks_column(pages.columns, main_month)], errors="coerce").sum()
    query_rows = int((pd.to_numeric(queries[metric_column(queries.columns, main_month, "Imp")], errors="coerce") > 0).sum())
    site_clicks = singapore[clicks_column(countries.columns, main_month)]
    workbooks.append((Path(book).name, main_month, filters_month, country_filter, len(countries), "/" in str(q_column),
                      query_clicks / site_clicks * 100, page_clicks / site_clicks * 100, query_rows))

workbooks = pd.DataFrame(workbooks, columns=["workbook", "main month", "date in Filters tab", "country filter",
                                             "countries listed", "date-style headers", "queries share %", "pages share %", "query rows"])
workbooks["mixes countries"] = workbooks["countries listed"] > 1
gsc = pd.DataFrame(gsc_rows, columns=["workbook", "month", "singapore clicks", "is main month"])
print("Workbooks read:", len(workbooks))

### Check, are all workbooks exported the same way?

In [ ]:
print("Country filter used:")
print(workbooks["country filter"].value_counts().to_string())
print()
print("Workbooks with no country filter written down:", workbooks.loc[workbooks["country filter"] == "none", "workbook"].tolist())
print("Workbooks whose tabs really mix countries:", workbooks.loc[workbooks["mixes countries"], "workbook"].tolist())
earlier = workbooks[workbooks["main month"] != workbooks["date in Filters tab"]]
print("Workbooks whose Filters tab date is the earlier of the two months in the headers:", earlier["workbook"].tolist())
print()
print("Column headers in the Queries tab:")
print(workbooks["date-style headers"].map({True: "dates, like 5/1/23 - 5/31/23 Clicks", False: "month names, like Jul Clicks"}).value_counts().to_string())

In [ ]:
extra_columns = set()
for book in books:
    for tab in ["Pages", "Search appearance"]:
        columns = pd.read_excel(book, sheet_name=tab, nrows=0).columns
        extra_columns.update(f"{tab}: {c}" for c in columns if str(c).startswith("Diff"))
print("Extra columns that only some workbooks have:", sorted(extra_columns))

In [ ]:
layout = pd.DataFrame(layout, columns=["tab", "columns"])
layout.groupby("tab").agg(workbooks=("columns", "size"), fewest_columns=("columns", "min"),
                          most_columns=("columns", "max")).sort_values("workbooks", ascending=False)

**How to read it.** The *Exclude* tabs show 0 columns only because their first row is empty. They hold Weavve's list of
words like *sateen weave* that look like the brand but are not (used below). If a tab always has the same layout, fewest and most columns are equal. Where they differ, the layout
changes between workbooks, and the header names change style too. So code must find columns by name and month, never by
position. One workbook was exported without the Singapore filter, so its tabs mix all countries. This notebook always
takes the Singapore row from the Countries tab, and treats the later of the two months in the headers as the workbook's main month.

### Explore, Singapore clicks per month

In [ ]:
copies = gsc.groupby("month")["singapore clicks"].agg(["count", "nunique"])
print("Rows:", len(gsc), "| different months:", gsc["month"].nunique())
print("Months that appear in two workbooks:", (copies["count"] > 1).sum(), "of", len(copies))
print("Months where the two copies disagree:", (copies["nunique"] > 1).sum())
spread = gsc.groupby("month")["singapore clicks"].agg(lambda c: (c.max() - c.min()) / c.max() * 100)
print(f"Largest gap between two copies: {spread.max():.1f}% ({spread.idxmax():%b %Y}). In every such month the later export is"
      " the same or higher, because an early export can miss the last days of the month.")
odd = gsc[gsc["month"] < "2022-12-01"]
print("Comparison months earlier than Dec 2022:", odd[["workbook", "month"]].values.tolist())

# One value per month: keep the later export (the copy where it is the earlier month), which covers the full month
ordered = gsc[gsc["month"] >= "2022-12-01"].sort_values("is main month", ascending=True)
monthly = ordered.drop_duplicates("month").set_index("month")["singapore clicks"].sort_index()
expected = pd.date_range(monthly.index.min(), monthly.index.max(), freq="MS")
print("Months covered:", monthly.index.min().strftime("%b %Y"), "to", monthly.index.max().strftime("%b %Y"),
      "| months with no value:", len(expected) - len(monthly))

In [ ]:
both = pd.DataFrame({"Search Console clicks (real)": monthly,
                     "Semrush visits (estimate)": semrush_month["estimated visits"]}).dropna()
line_chart(both, "Real clicks vs Semrush estimate, per month", ylabel="clicks or visits", colors=[GREEN, ORANGE])

peak = monthly.idxmax()
pearson = both.corr().iloc[0, 1]
pearson_no_peak = both.drop(index=peak).corr().iloc[0, 1]
spearman = both.rank().corr().iloc[0, 1]
ratio = (both.iloc[:, 1] / both.iloc[:, 0]).median()
print(f"Correlation {pearson:.2f}, without {peak:%b %Y} {pearson_no_peak:.2f}, by rank {spearman:.2f}")
print(f"Semrush's estimate is typically {ratio:.1f} times the real clicks")

**How to read it.** Correlation runs from -1 to 1. Near 1 means the two lines rise and fall together, near 0 means they don't.
One huge month can drag it down, so it is shown with and without that month, and by rank, which ignores the size of the jumps.

In [ ]:
peak_book = gsc.loc[(gsc["month"] == peak) & gsc["is main month"], "workbook"].iloc[0]
pages = pd.read_excel(next(b for b in books if Path(b).name == peak_book), sheet_name="Pages")
peak_column = clicks_column(pages.columns, peak)
print("Peak month:", peak.strftime("%b %Y"), "with", format(monthly.max(), ","), "clicks")
top_pages = pages.nlargest(5, peak_column)[["Top pages", peak_column]].copy()
top_pages["share of the month %"] = (top_pages[peak_column] / monthly.max() * 100).round(1)
top_pages["share of the Pages tab %"] = (top_pages[peak_column] / pd.to_numeric(pages[peak_column], errors="coerce").sum() * 100).round(1)
pd.set_option("display.max_colwidth", 100)    # show the full page address
top_pages

### Check, how complete are the Queries and Pages tabs?

In [ ]:
coverage_tabs = workbooks.sort_values("main month").set_index("main month")[["query rows", "queries share %", "pages share %"]].round(1)
print("Queries listed for the month: exactly 1,000 up to", coverage_tabs[coverage_tabs["query rows"] == 1000].index.max().strftime("%b %Y"),
      "(an export limit), then", coverage_tabs.loc[coverage_tabs.index > "2024-04-30", "query rows"].min(), "to",
      coverage_tabs.loc[coverage_tabs.index > "2024-04-30", "query rows"].max())
print("Workbooks whose Pages tab holds under 40% of the month's clicks (probably cut off):")
coverage_tabs[coverage_tabs["pages share %"] < 40]

### Explore, branded and non-branded searches over time

Weavve's own summary in the Filters tab counts a search as **branded** if it contains *wea*, except for the words in its Exclude
list (like *sateen weave*). Using that same rule for every month shows whether the drop in search clicks comes from people
searching for the brand, or from everything else.

In [ ]:
exclude = set()
for book in books:
    workbook = pd.ExcelFile(book)
    for tab in workbook.sheet_names:
        if tab.lower().startswith("exclude"):
            exclude.update(workbook.parse(tab, header=None).iloc[:, 0].dropna().astype(str).str.lower().str.strip())

brand_rows = []
for _, row in workbooks[~workbooks["mixes countries"]].iterrows():
    q = pd.read_excel(next(b for b in books if Path(b).name == row["workbook"]), sheet_name="Queries")
    column = clicks_column(q.columns, row["main month"])
    text = q["Top queries"].astype(str).str.lower().str.strip()
    is_brand = text.str.contains("wea") & ~text.isin(exclude)
    clicks = pd.to_numeric(q[column], errors="coerce")
    brand_rows.append((row["main month"], clicks[is_brand].sum(), clicks[~is_brand].sum()))
brand_split = pd.DataFrame(brand_rows, columns=["month", "branded", "non-branded"]).groupby("month").last()
line_chart(brand_split, "Query clicks by type, Weavve's own brand rule", ylabel="clicks", colors=[GREEN, ORANGE])
latest_month = brand_split.index.max()
year_before = latest_month - pd.DateOffset(years=1)
if year_before in brand_split.index:
    change = (brand_split.loc[latest_month] / brand_split.loc[year_before] - 1) * 100
    print(f"{latest_month:%b %Y} against {year_before:%b %Y}: branded {change['branded']:+.0f}%, non-branded {change['non-branded']:+.0f}%")
print("Branded share of query clicks, latest month:", round(brand_split.loc[latest_month, "branded"] / brand_split.loc[latest_month].sum() * 100, 1), "%")

**How to read it.** These are clicks from the Queries tab, which lists about half of all clicks, so the levels are lower than the
site total but the trend is comparable. The spikes are the new-year blog post. Branded clicks hold fairly steady, while non-branded
clicks, people finding Weavve through general searches, have fallen. That is where the organic decline sits.

### Check, does the Queries tab hold all the clicks?

In [ ]:
filtered_books = workbooks[~workbooks["mixes countries"]]
print("Share of the month's Singapore clicks found in the Queries tab, one value per workbook (Singapore-only workbooks)")
numbers(filtered_books["queries share %"]).to_frame("%")

**How to read it.** Google leaves rare searches out of the Queries tab for privacy, and older exports were also capped at 1,000
queries, so the tab only holds part of the clicks.

**What this tells us about Search Console**

- The workbooks were not all exported the same way. One mixes all countries, file names list the two months in different orders,
  and headers and layouts change between workbooks. For example, some Pages tabs add extra *Diff* columns.
- Most months appear in two workbooks. Some copies differ by a few percent because an early export can miss the last days of the
  month, so this notebook keeps the later export.
- One workbook, *Mar 2023 vs Apr 2023*, actually compares Apr 2023 with **Mar 2022**, a year earlier, so its difference columns
  mean something different from every other workbook. Its Mar 2022 figures are left out of the monthly series.
- In the biggest month, one blog post about lucky colours for the new year brought at least 41% of all clicks
  (70% of the clicks the Pages tab lists).
  Clicks should be split into shopping and non-shopping searches before judging the SEO trend.
- The Queries tab holds only about half of the clicks, so query-level totals are always lower than the site total.
- Semrush's estimate runs well above the real clicks. The two lines only partly move together, so the real clicks are
  the ones to trust when they differ, as the project instructions warn.
- By Weavve's own brand rule, about half of recent query clicks come from brand searches. Brand searches hold steady while
  non-branded clicks have fallen, which locates the organic decline.
- The Queries tab was capped at 1,000 queries until May 2024, and one recent Pages tab is cut off, so counts of queries or pages
  should not be trended across those points.

## I. Every column in every tab

Sections B to H look closely at the columns that matter for the project. To make sure nothing is missed, this section runs the
same seven checks on **every column of every tab** in all 7 files, 23 tabs in total.

| Check | What it catches |
|---|---|
| all empty | a column with no values at all |
| one value only | every filled cell holds the same value, so the column tells us nothing |
| mostly empty | more than half the cells are empty |
| extra spaces | text with a space at the start or end, so *King* and *King␣* count as two different values |
| case or space variants | values that differ only by capitals or spaces, such as *Staff discount* and *STAFF DISCOUNT* |
| numbers stored as text | numbers kept as text, which breaks sums and averages |
| mixed text and numbers | one column holding both text and numbers |

Search Console has 38 workbooks, so its columns are checked in every workbook and the month in each header is replaced by the word *month*.

In [ ]:
def column_flags(values):
    """The seven checks for one column. Returns a list of findings, empty if nothing was found."""
    filled = values.dropna()
    if len(filled) == 0:
        return ["all empty"]
    found = []
    if len(filled) > 1 and filled.astype(str).nunique() == 1:
        found.append("one value only")
    empty_share = 1 - len(filled) / len(values)
    if empty_share > 0.5:
        found.append(f"mostly empty ({empty_share:.0%})")
    text = filled[filled.map(type) == str]
    if len(text) > 0:
        spaces = int((text != text.str.strip()).sum())
        if spaces:
            found.append(f"extra spaces ({spaces:,})")
        variants = text.nunique() - text.str.strip().str.lower().nunique()
        if variants:
            found.append(f"case or space variants ({variants:,})")
        numeric_text = int(pd.to_numeric(text.str.replace(",", "").str.strip(), errors="coerce").notna().sum())
        if numeric_text:
            found.append(f"numbers stored as text ({numeric_text:,})")
        if len(text) < len(filled):
            found.append("mixed text and numbers")
    return found


def plain_header(header):
    """Replace the month in a Search Console header with the word 'month', so the column matches across workbooks."""
    header = re.sub(r"\d{1,2}/\d{1,2}/\d{2,4}\s*-\s*\d{1,2}/\d{1,2}/\d{2,4}", "month", str(header).strip())
    header = re.sub(r"^(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*( \d{4})? ", "month ", header)
    return header.replace("Impressions", "Imp").replace("Position", "Pos")


# every tab of files 2 to 12, read fresh so the columns this notebook added are not included
tables = [("3 Orders", "Order Transaction-Level", load.orders()),
          ("4 Products", "ProductExport", load.skus()),
          ("9.1 Loyalty", "Redemptions", load.loyalty()),
          ("2 Semrush", "all 43 monthly files", load.semrush().drop(columns=["month", "source_file"]))]
for key, file in [("discounts", "8 Discounts"), ("ads", "12 Google Ads")]:
    book = pd.ExcelFile(paths.path(key))
    for tab in book.sheet_names:
        tables.append((file, tab.strip(), book.parse(tab)))

tab_rows, found_rows = [], []
for file, tab, table in tables:
    flagged = 0
    for column in table.columns:
        found = column_flags(table[column])
        if found:
            flagged += 1
            found_rows.append((file, tab, str(column), "; ".join(found)))
    tab_rows.append((file, tab, f"{len(table):,}", table.shape[1], flagged))

# every tab of every Search Console workbook
gsc_tabs, gsc_found, pages_blank = {}, {}, {}
for book in books:
    for tab, table in pd.read_excel(book, sheet_name=None).items():
        if tab == "Pages" and table["Top pages"].isna().sum() > 100:
            pages_blank[Path(book).name] = int(table["Top pages"].isna().sum())   # used in the next table
        seen = gsc_tabs.setdefault(tab, {"workbooks": 0, "rows": 0, "columns": set(), "flagged": set()})
        seen["workbooks"] += 1
        seen["rows"] += len(table)
        for column in table.columns:
            name = plain_header(column)
            seen["columns"].add(name)
            for finding in column_flags(table[column]):
                kind = finding.split(" (")[0]
                spot = gsc_found.setdefault((tab, kind), {"workbooks": set(), "columns": set()})
                spot["workbooks"].add(Path(book).name)
                spot["columns"].add(name)
                seen["flagged"].add(name)
for (tab, kind), spot in sorted(gsc_found.items()):
    found_rows.append(("1 Search Console", tab, ", ".join(sorted(spot["columns"])),
                       f"{kind} (in {len(spot['workbooks'])} of {gsc_tabs[tab]['workbooks']} workbooks)"))
for tab, seen in gsc_tabs.items():
    tab_rows.append(("1 Search Console", f"{tab} ({seen['workbooks']} of {len(books)} workbooks)", f"{seen['rows']:,}",
                     len(seen["columns"]), len(seen["flagged"])))

file_order = lambda s: s.str.split(" ").str[0].astype(float) if s.name == "file" else s
every_tab = pd.DataFrame(tab_rows, columns=["file", "tab", "rows", "columns", "columns with a finding"])
every_tab = every_tab.sort_values(["file", "tab"], key=file_order).reset_index(drop=True)
findings = pd.DataFrame(found_rows, columns=["file", "tab", "column", "what the check found"])
findings = findings.sort_values("file", key=file_order, kind="stable").reset_index(drop=True)
print(f"Checked {len(every_tab)} tabs and {every_tab['columns'].sum():,} columns. "
      f"{len(findings)} lines below have at least one finding.")
every_tab

**How to read it.** One line per tab. For Search Console, `rows` adds up every workbook and `columns` counts each column
name once. The last column says how many of the tab's columns have at least one finding, listed in the next table.

In [ ]:
pd.set_option("display.max_rows", 200)
findings

**How to read it.** Only columns with a finding are listed. Most were already covered in sections B to H, such as the empty and
fixed columns in Products and Loyalty, `Discount code` being empty on most rows, and Brand 0. Some are harmless:
*Applies Once Per Customer* is 1 or blank, where blank means no. The Countries tab has one row, and the Filters tab mixes text
and numbers because it is a list of settings. Keywords such as *2025* show as numbers stored as text in Semrush, which is correct.

In [ ]:
# The findings that sections B to H did not cover, measured

raw_ads = pd.ExcelFile(paths.path("ads")).parse("Search keyword report All time ")
raw_orders = load.orders()
staff_names = raw_orders["Discount name"].dropna()
staff_names = staff_names[staff_names.str.lower().str.contains("staff")]
sizes = products["Option 1"].dropna()
sizes = sizes[sizes.map(type) == str]

pd.DataFrame([
    ("1 Search Console", "Pages tab has blank rows after the last page, where only the Diff formula shows 0",
     f"{len(pages_blank)} workbooks, up to {max(pages_blank.values()):,} blank rows"),
    ("1 Search Console", "An extra tab called Sheet1: someone's side comparison of 9 blog pages, with a total row",
     f"{gsc_tabs['Sheet1']['workbooks']} workbook"),
    ("12 Google Ads", "Max. CPC (the bid) is text such as '--' or '2.00 (enhanced)'",
     f"{(raw_ads['Max. CPC'].astype(str).str.strip() == '--').sum():,} rows of '--'"),
    ("12 Google Ads", "Interaction rate is '--' when a keyword had no impressions",
     f"{(raw_ads['Interaction rate'].astype(str).str.strip() == '--').sum():,} rows"),
    ("12 Google Ads", "The same keyword written with different capitals, such as 'Duvet' and 'duvet'",
     f"{raw_ads['Keyword'].nunique() - raw_ads['Keyword'].str.lower().nunique()} keywords"),
    ("3 Orders", "Return reason is filled on very few rows",
     f"{raw_orders['Return reason'].notna().sum()} of {len(raw_orders):,} rows"),
    ("3 Orders", "Discount names written in different ways, such as 'STAFF', 'staff ' and 'Staff Purchase'",
     f"{staff_names.nunique()} spellings of staff discounts alone"),
    ("4 Products", "Values in Option 1 (size or colour) with extra spaces, such as 'Queen ' next to 'Queen'",
     f"{(sizes != sizes.str.strip()).sum()} products"),
], columns=["file", "finding", "size"])

**What the full check adds**

- **Search Console Pages tabs** in 6 recent workbooks end with hundreds of blank rows. Totals are not affected, but pages must be
  counted by page name, not by row.
- **Sheet1** in one workbook is not a Google export. It is ignored.
- **Google Ads bids** (`Max. CPC`) are text and need cleaning before any bid analysis. Keywords that differ only by capitals are the
  same keyword to Google, so the notebook groups them in lower case (the `kw` column).
- **Return reasons** are almost never recorded, so the project cannot study why customers return items.
- **Discount names and product sizes** need spaces removed and capitals made the same before grouping. Otherwise *Staff Purchase*
  and *STAFF PURCHASE* are counted as two discounts, and *King* and *King␣* as two sizes.

## J. Summary

### Coverage, what period each file covers

In [ ]:
pd.DataFrame([
    ("1 Search Console", monthly.index.min().strftime("%b %Y"), monthly.index.max().strftime("%b %Y")),
    ("2 Semrush", semrush["month"].min().strftime("%b %Y"), semrush["month"].max().strftime("%b %Y")),
    ("3 Orders", orders["Day"].min().strftime("%d %b %Y"), orders["Day"].max().strftime("%d %b %Y")),
    ("4 Products", "a current list, no dates", ""),
    ("8 Discounts", start.min().strftime("%b %Y"), start.max().strftime("%b %Y")),
    ("9.1 Loyalty", loyalty["date"].min().strftime("%d %b %Y"), loyalty["date"].max().strftime("%d %b %Y")),
    ("12 Google Ads", "all time totals, no dates per row", ""),
], columns=["file", "from", "to"])

**How to read it.** Orders, Search Console and Semrush all cover **Jan 2023 to Jul 2026**, so comparisons across these files should use that period.
Loyalty only starts in Apr 2023.

### Issues found, most serious first

In [ ]:
pct = lambda part, whole: f"{part / whole * 100:.1f}%"
issues = pd.DataFrame([
    # severity, file, issue, size, impact on the analysis, how the team will handle it
    ("High", "3 Orders", "Undo rows and cancelled orders sit with sales",
     f"{(~SALE).sum():,} undo rows, {CANCELLED.sum():,} rows on cancelled orders",
     "Sales and order counts wrong in either direction, depending on which rows are kept",
     "Keep sale rows of orders not cancelled, subtract real refunds"),
    ("High", "3 Orders", "Items with stacked discounts written twice, full Net sales on each",
     f"{extra_rows:,} extra rows, {pct(extra_value, sales['Net sales'].sum())} of sales value",
     "Sales, order values and items per order overstated", "Count each order line once"),
    ("High", "3 Orders", "Marketplace customer IDs almost never repeat",
     f"{orders_per_id['Marketplace'].mean():.2f} orders per marketplace ID",
     "Repeat rate and customer value far too low for marketplace customers", "Measure repeat buying on the website only"),
    ("High", "3 Orders", "One customer ID is the Osco channel",
     f"{orders['Customer number of orders'].max():,} orders",
     "Inflates concentration of value and repeat rates", "Set it aside in customer analysis"),
    ("High", "3 Orders and 8 Discounts", "Discount details (code, name, title, value) only exported from 1 Jan 2023",
     "every channel", "Which discount was used is unknown before 2023",
     "Study codes from 2023, use the Discounts amount before"),
    ("High", "1 Search Console", "A workbook not filtered to Singapore",
     f"{workbooks['mixes countries'].sum()} workbook", "Clicks for that month overstated if the first row is taken",
     "Always take the Singapore row"),
    ("High", "1 Search Console", "Non-branded search clicks falling while brand clicks hold",
     "see the brand chart in section H", "The SEO decline is in general searches, not brand ones", "Report branded and non-branded separately"),
    ("High", "2 Semrush", "Old rows carried into monthly files from Sep 2025",
     f"up to {carried.max():.0f}% of a month's rows", "Recent keyword counts and visits overstated", "Use fresh rows only"),
    ("High", "12 Google Ads", "Half of conversions carry almost no value",
     "2 campaigns", "Keyword results overstated by conversion counts", "Judge keywords on conversion value"),
    ("High", "3 Orders", "Marketplace Net sales include GST from 26 Aug 2025",
     "about 9% on those rows", "Recent marketplace sales overstated", "Divide those rows by 1.09"),
    ("Medium", "3 Orders", "Many marketplace discounts are price match or platform funded", f"{mp['price match'].sum():,} and {mp['platform funded'].sum():,} lines",
     "Marketplace discount dependence overstated", "Report with and without them"),
    ("Medium", "3 Orders", "Lifetime columns count cancelled orders, on Total sales", "3 columns",
     "Customer frequency and spend not comparable with Net sales", "Rebuild customer measures from kept orders"),
    ("High", "1 Search Console", "One blog post inflates the biggest month",
     f"{top_pages.iloc[0, 2]}% of {peak:%b %Y} clicks", "SEO trend looks stronger than it is around that month",
     "Split shopping and non-shopping searches"),
    ("Medium", "4 Products", "Items sold that are not in the product list",
     f"{join.loc['code NOT in the product list', '% of value']}% of sales value, plus {join.loc['no code at all', '% of value']}% with no code",
     "Category sales understated, mostly for old products", "Flag them, try matching on the start of the code"),
    ("Medium", "8 Discounts", "Per-customer copies mixed with evergreen codes",
     f"{(family != 'Other').sum():,} of {len(discounts):,} entries are copies",
     "Counting codes overstates how many campaigns ran", "Group entries into families"),
    ("Low", "8 Discounts", "Codes in Orders that match no report entry, even by campaign prefix",
     f"{(~explained).mean() * 100:.1f}% of website code rows", "Discount details missing for those orders",
     "Match by name, then by campaign prefix"),
    ("Medium", "3 Orders", "A few very large orders",
     f"{numbers(per_order['value']).iloc[-1]} orders above the unusually-high line",
     "Averages pulled up", "Report medians, check the largest orders"),
    ("Medium", "1 Search Console", "Layouts and header names change between workbooks",
     "Pages, Search appearance, Queries headers", "Code reading by position picks the wrong column",
     "Find columns by name and month"),
    ("Medium", "1 and 2", "Semrush estimates differ from real clicks",
     f"about {ratio:.1f} times higher", "Traffic overstated if Semrush is used", "Use Search Console where they differ"),
    ("Low", "1 Search Console", "Months repeated in two workbooks, some differ slightly",
     f"{(copies['nunique'] > 1).sum()} months, largest gap {spread.max():.1f}%", "Small differences in monthly clicks",
     "Keep the later, full-month export"),
    ("Low", "1 Search Console", "Filters tab date differs from the column headers",
     f"{len(earlier)} workbook", "Wrong month if the Filters tab is trusted", "Take the later month in the headers"),
    ("Low", "1 Search Console", "One workbook compares with the same month a year earlier",
     "Mar 2023 vs Apr 2023 workbook", "Its difference columns are not month-on-month", "Leave its Mar 2022 figures out"),
    ("Low", "1 Search Console", "Queries tab holds only part of the clicks",
     f"median {filtered_books['queries share %'].median():.0f}%", "Query-level totals understated",
     "Use the Countries total for site clicks"),
    ("Low", "1 Search Console", "Queries tab capped at 1,000 rows until May 2024; one Pages tab cut off",
     "see section H", "Query and page counts not comparable over time", "Do not trend query or page counts"),
    ("Low", "12 Google Ads", "Spend covers search keywords only", f"{len(only_in_negatives)} other campaigns",
     "Total Google Ads spend understated", "State the scope"),
    ("Low", "2 Semrush", "Keywords repeated within a month",
     f"{semrush.duplicated(['month', 'Keyword']).sum():,} repeated rows", "Keyword counts overstated", "Count keywords once"),
    ("Low", "3 Orders", "0 used instead of empty", "Refund ID, Line item ID", "Missing values hidden", "Treat 0 as missing"),
    ("Low", "4 Products", "Brand is 0 instead of a name", f"{(products['Brand'].astype(str) == '0').sum()} product codes",
     "Brand analysis incomplete", "Treat 0 as missing"),
    ("Low", "8 Discounts", "Test codes in the report", f"{names.str.contains('test', case=False).sum()} entries",
     "Small noise in discount counts", "Leave out of discount analysis"),
    ("Medium", "9.1 Loyalty", "Points vouchers before Mar 2025 gave the other reward's value ($50 label gave $90, and back)", f"{period.str.startswith('1').sum():,} vouchers",
     "Reward value and points cost wrong", "Take value from the discount report"),
    ("Medium", "8 Discounts", "Report is a snapshot, Orders run later", f"report to {start_all.max():%d %b %Y}",
     "Order counts above report counts", "Stop comparisons at the report date"),
    ("Low", "all", "Files 6, 7, 10 and 11 not in data/ yet", "4 files", "Weavve's own customer summaries not used as a check",
     "Download from eLearn"),
    ("Medium", "3 Orders, 4 Products", "Same discount name or size written with different capitals or spaces",
     f"{staff_names.nunique()} spellings of staff discounts alone", "One discount or size counted as several",
     "Remove spaces and use one case before grouping"),
    ("Low", "1 Search Console", "Blank rows at the end of the Pages tab, and a stray Sheet1 tab",
     f"{len(pages_blank)} workbooks, 1 workbook", "Page counts too high if rows are counted", "Count pages by name, ignore Sheet1"),
    ("Low", "12 Google Ads", "Bid column (Max. CPC) stored as text with '--' and '(enhanced)'", "every row",
     "Bids cannot be summed or compared", "Strip the text, treat '--' as missing"),
    ("Low", "3 Orders", "Return reason almost never recorded", f"{raw_orders['Return reason'].notna().sum()} rows",
     "Reasons for returns cannot be studied", "Leave out of scope"),
], columns=["severity", "file", "issue", "size", "impact on the analysis", "how the team will handle it"])
issues = issues.sort_values("severity", key=lambda s: s.map({"High": 0, "Medium": 1, "Low": 2}), kind="stable").reset_index(drop=True)
issues

**The three that matter most.** Orders mixes undo rows, cancelled orders and double-written discount rows with real sales,
so every sales figure depends on cleaning them first. Marketplace customer IDs cannot follow a shopper, so customer analysis
must be done on the website. And discount codes were only recorded from 2023, so discount analysis by code covers three and a half years at most.